In [0]:
# 라이브러리 불러오기
import os
import random
import mlflow.deployments
from langchain.text_splitter import RecursiveCharacterTextSplitter
from mlflow.deployments import get_deploy_client
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col

In [0]:
# 문서 불러오기
volume_path = "/Volumes/brickstore/bronze/documents"

documents = []
for filename in os.listdir(volume_path):
    if filename.endswith(".md"):
        with open(os.path.join(volume_path, filename), "r") as f:
            content = f.read()
            documents.append({"filename": filename, "content": content})
            print(f"✓ {filename}: {len(content)} chars")

print(f"{len(documents)}개 문서 로딩 완료")

In [0]:
# 텍스트 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,         # 각 청크의 최대 문자 수
    chunk_overlap=50,       # 청크 간 겹치는 문자 수 (문맥 유지)
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]  # 분할 우선순위
)

chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc["content"])
    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['filename']}_{i}",
            "source": doc["filename"],
            "content": chunk
        })

print(f"총 {len(chunks)}개 청크 생성")
print(f"\n--- 예시 청크 ---")
print(f"Source: {chunks[10]['source']}")
print(f"Content: {chunks[10]['content'][:200]}...")

In [0]:
# 청크를 Delta Table로 저장
schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("source", StringType(), False),
    StructField("content", StringType(), False),
])

silver_chunks = spark.createDataFrame(chunks, schema)
silver_chunks.write.mode("overwrite").saveAsTable("brickstore.silver.document_chunks")
spark.sql("ALTER TABLE brickstore.silver.document_chunks SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"✓ workshop.ecommerce.document_chunks: {silver_chunks.count()} rows")
display(silver_chunks.limit(5))

In [0]:
# Embedding 모델 목록 확인
deploy_client = get_deploy_client('databricks')
embedding_models = [ep['name'] for ep in deploy_client.list_endpoints() if ep['task'] == 'llm/v1/embeddings']
print(embedding_models)

In [0]:
# 한글을 지원하는 모델의 상세정보 확인
display(deploy_client.get_endpoint("databricks-qwen3-embedding-0-6b"))

In [0]:
# Amazon Bedrock의 Claude 모델을 외부 제공자로 등록
model_name = "global.anthropic.claude-opus-4-6-v1"
aws_region = spark.conf.get("spark.databricks.clusterUsageTags.region")

if model_name not in [endpoints['name'] for endpoints in deploy_client.list_endpoints()]:
    deploy_client.create_endpoint(
        name="bedrock-claude-opus-4-6",
        config={
            "served_entities": [
                {
                    "external_model": {
                        "name": model_name,
                        "provider": "amazon-bedrock",
                        "task": "llm/v1/chat",
                        "amazon_bedrock_config": {
                            "aws_region": aws_region,
                            "uc_service_credential_name": "databricks-bedrock-role",
                            "bedrock_provider": "anthropic",
                        },
                    }
                }
            ]
        },
    )
else:
    print("model already created")